# 🚧 Pothole Detection — YOLOv11 Training (Foolproof Version)

**This notebook has been completely automated.**
You do **NOT** need to attach any datasets manually. It will automatically download a public pothole dataset from GitHub and train on it!

### Steps:
1. Go to **Settings -> Accelerator -> GPU T4 x2** to enable the GPU.
2. Click **Run All**!
3. Wait 2 hours.
4. Download `best.pt` from the right sidebar (`working/runs/pothole_yolo11/weights/best.pt`).

In [ ]:
# ── Step 1: Install Dependencies & Prevent Crashes ────────────────────────
!pip install "numpy<2.0.0" ultralytics albumentations -q
print('✅ Dependencies installed (Numpy crash prevented!)')

In [ ]:
# ── Step 2: GPU Check ─────────────────────────────────────────────────────
import torch
print('=' * 50)
print(f'  CUDA available : {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'  GPU            : {torch.cuda.get_device_name(0)}')
else:
    print('  ⚠️  No GPU — Go to Settings -> Accelerator -> GPU T4 x2')
print('=' * 50)

In [ ]:
# ── Step 3: Automatically Download Dataset (Bypassing Kaggle UI) ──────────
!wget -q -O pothole_dataset.zip https://github.com/jaygala24/pothole-detection/releases/download/v1.0.0/Pothole.Dataset.IVCNZ.zip
!unzip -q pothole_dataset.zip -d /kaggle/working/downloaded_dataset
print('✅ Dataset downloaded and extracted successfully!')

In [ ]:
# ── Step 4: Organise Dataset ──────────────────────────────────────────────
import os, shutil, random
from pathlib import Path

INPUT_DIR = Path('/kaggle/working/downloaded_dataset')
MERGED_DIR = Path('/kaggle/working/merged_dataset')

for split in ['train', 'val', 'test']:
    (MERGED_DIR / split / 'images').mkdir(parents=True, exist_ok=True)
    (MERGED_DIR / split / 'labels').mkdir(parents=True, exist_ok=True)

print("1. Finding all label files...")
labels_map = {}
for p in INPUT_DIR.rglob('*.txt'):
    if p.name.lower() not in ['readme.txt', 'classes.txt', 'requirements.txt']:
        labels_map[p.stem] = p

print("2. Finding images and matching...")
EXTS = {'.jpg', '.jpeg', '.png'}
pairs = []
for img_p in INPUT_DIR.rglob('*'):
    if img_p.suffix.lower() in EXTS:
        if img_p.stem in labels_map:
            pairs.append((img_p, labels_map[img_p.stem]))

print(f"✅ Found {len(pairs)} valid image-label pairs!")

if len(pairs) == 0:
    raise ValueError("❌ Dataset extraction failed or structure is unrecognized.")

# Shuffle and split
random.shuffle(pairs)
n = len(pairs)
cuts = [int(n * 0.75), int(n * 0.90)]
splits = {
    'train': pairs[:cuts[0]],
    'val':   pairs[cuts[0]:cuts[1]],
    'test':  pairs[cuts[1]:]
}

def remap_label_to_0(label_path, out_path):
    lines = []
    try:
        with open(label_path) as f:
            for line in f:
                parts = line.strip().split()
                if len(parts) >= 5:
                    parts[0] = '0'
                    lines.append(' '.join(parts))
    except Exception: return False
    if lines:
        with open(out_path, 'w') as f:
            f.write('\n'.join(lines))
        return True
    return False

added = 0
for split, sp in splits.items():
    for i, (ip, lp) in enumerate(sp):
        name = f'pothole_{i:05d}'
        dst_img = MERGED_DIR / split / 'images' / (name + ip.suffix)
        dst_lbl = MERGED_DIR / split / 'labels' / (name + '.txt')
        shutil.copy2(ip, dst_img)
        if remap_label_to_0(lp, dst_lbl):
            added += 1

print(f"\n🎉 Successfully copied {added} pairs to /kaggle/working.")
for split in ['train', 'val', 'test']:
    c = len(list((MERGED_DIR / split / 'images').glob('*')))
    print(f'  {split}: {c} images')

In [ ]:
# ── Step 5: Write dataset.yaml ────────────────────────────────────────────
import yaml

yaml_path = MERGED_DIR / 'dataset.yaml'
cfg = {
    'path':  str(MERGED_DIR),
    'train': 'train/images',
    'val':   'val/images',
    'test':  'test/images',
    'nc':    1,
    'names': ['pothole'],
}
with open(yaml_path, 'w') as f:
    yaml.dump(cfg, f, default_flow_style=False)

print('✅ dataset.yaml created:')
print(yaml.dump(cfg))

In [ ]:
# ── Step 6: TRAIN YOLOv11 ───────────────────────────────────────────
# ⏱️ This takes a couple of hours. Do NOT close the tab!

from ultralytics import YOLO

model = YOLO('yolo11m.pt')  # YOLOv11 Medium

results = model.train(
    data         = str(yaml_path),
    epochs       = 100,
    imgsz        = 640,
    batch        = 16,
    optimizer    = 'AdamW',
    lr0          = 0.001,
    patience     = 20,
    save         = True,
    project      = '/kaggle/working/runs',
    name         = 'pothole_yolo11',
    device       = 0,
)

print('\n✅ Training complete!')
print(f'Weights saved at: {results.save_dir}/weights/best.pt')